### Cell 1 -- Install (lean, no restart, no version pins)

In [1]:
import subprocess, sys

# No hard version pins: Unsloth moves fast and pinning transformers/trl/peft to specific
# versions here risks fighting whatever unsloth_zoo's *current* release actually expects
# (mismatched pins cause exactly the kind of "NameError inside a compatibility patch" error
# you'd get from a stale transformers version paired with a fresh unsloth_zoo). Let pip's
# resolver pick versions that are mutually compatible right now.
print("Installing/upgrading Unsloth + RL stack...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "--no-cache-dir",
                        "unsloth", "unsloth_zoo", "trl", "peft", "transformers",
                        "accelerate", "bitsandbytes"])
print("Ready.")


Installing/upgrading Unsloth + RL stack...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 353.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 354.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 404.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 357.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 358.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 239.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 290.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 345.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 273.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 342.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 335.5 MB/s eta 0:00:00
 

### Cell 2 -- Load model via Unsloth

A 1B model instead of 3B: on a free T4 this is the difference between fitting the whole run in ~30 minutes vs. not. Swap `MODEL_ID` for the 3B if you have more time budget.

**No new special tokens, no embedding resize.** `<state>` is plain text -- it tokenizes into existing, already-trained subword pieces, so it's learnable from step 1 through ordinary LoRA on the attention/MLP projections. This fixes the frozen-embedding bug in the original.

In [2]:
# Guard against a family of known open Unsloth import-time bugs (e.g. unslothai/unsloth#3825):
# various startup checks inside Unsloth call importlib.util.find_spec("some_optional_pkg")
# (vllm, fbgemm_gpu, etc.), which raises "ValueError: X.__spec__ is None" if the environment has
# a broken/placeholder entry for that package in sys.modules -- unrelated to anything in this
# notebook. We don't need to know *why* any given one is broken; we make find_spec fail soft for
# the whole duration of the import (covers all of them at once), then restore it.
#
# Self-healing, safe to re-run WITHOUT restarting the kernel: importlib.util is reloaded first,
# which re-defines find_spec fresh from its original source and discards any monkeypatch left
# behind by an earlier attempt in this same session (a stale patch from a prior failed run is
# exactly what caused a RecursionError here before -- reloading removes that possibility instead
# of trusting whatever find_spec currently happens to be).
import importlib
import importlib.util as _ilu

importlib.reload(_ilu)
_pristine_find_spec = _ilu.find_spec

def _find_spec_soft(name, *a, **kw):
    try:
        return _pristine_find_spec(name, *a, **kw)
    except ValueError:
        return None

_ilu.find_spec = _find_spec_soft
try:
    from unsloth import FastLanguageModel
finally:
    _ilu.find_spec = _pristine_find_spec

import torch

MODEL_ID = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"   # swap to the 3B variant if you have more time
max_seq_length = 384  # short prompts + short completions -> faster generation, fits the time budget

print("Loading model via Unsloth (4-bit)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

print("Applying LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model ready.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model via Unsloth (4-bit)...
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.


Applying LoRA...


Unsloth 2026.7.2 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Model ready.


### Cell 3 -- The ground-truth affect reader

This is the actual scientific grounding. `j-hartmann/emotion-english-distilroberta-base` is a real, published (Hartmann, 2022) 7-class emotion classifier -- not something invented for this notebook. Its categorical outputs are mapped to continuous Valence/Arousal using placements broadly supported by the affect-circumplex literature (Russell, 1980; Posner, Russell & Peterson, 2005). It's the one thing in this notebook the model's self-reports get checked against, and it runs on CPU so it doesn't compete with the LLM for GPU time.

In [3]:
from transformers import pipeline

emotion_clf = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=-1,   # CPU: keeps the GPU free for the LLM, plenty fast for short texts
)

# Approximate circumplex placements (valence, arousal), each in [0, 1].
# These are simplifications of a well-studied mapping, not lab-grade precision numbers.
EMOTION_VAD = {
    "joy":      (0.90, 0.70),
    "surprise": (0.60, 0.80),
    "neutral":  (0.50, 0.35),
    "sadness":  (0.20, 0.30),
    "disgust":  (0.15, 0.55),
    "fear":     (0.20, 0.75),
    "anger":    (0.15, 0.80),
}

def measure_affect(text: str) -> tuple[float, float]:
    """Real, external, reproducible (valence, arousal) estimate for a piece of text."""
    text = (text or "").strip()
    if not text:
        return (0.5, 0.5)
    scores = emotion_clf(text[:512])[0]  # list of {label, score}
    v = sum(EMOTION_VAD[s["label"]][0] * s["score"] for s in scores)
    a = sum(EMOTION_VAD[s["label"]][1] * s["score"] for s in scores)
    return (v, a)

print("Affect reader ready:", measure_affect("I just got promoted after 3 years of hard work!"))


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Affect reader ready: (0.7137660234235227, 0.7260905842413194)


### Cell 4 -- Build GRPO dataset

Uses `apply_chat_template` instead of a hand-written template string, which avoids the duplicate-BOS bug. The system prompt is honest about what the model is doing -- structured self-appraisal, not literal biology.

In [4]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "Before replying, briefly estimate the emotional tone of the user's message using two numbers: "
    "Valence (0 = very negative, 1 = very positive) and Arousal (0 = very calm, 1 = very intense). "
    "This is the psychological Valence-Arousal model of affect, not a claim about anything physical. "
    "Write it as:\n<state>Valence: X.X, Arousal: X.X</state>\n"
    "Then write your reply, in a tone consistent with that appraisal."
)

RAW_PROMPTS = [
    "I just got promoted after 3 years of hard work!",
    "My dog passed away this morning.",
    "You are completely useless and I hate talking to you.",
    "Can you explain what photosynthesis is?",
    "I've been feeling really anxious lately and I don't know why.",
    "We just closed the biggest deal in our company's history!",
    "I'm deeply, existentially bored.",
    "What is 15% of 340?",
    "I feel like nobody truly understands me.",
    "This is literally the best day of my life!",
    "I'm scared about my surgery tomorrow.",
    "Let's collaborate on something creative together.",
    "I've been betrayed by someone I trusted completely.",
    "Thank you so much, you really helped me today.",
    "I don't care anymore. About anything.",
    "Can you help me write a cover letter?",
    "My partner just broke up with me out of nowhere.",
    "I just finished my first 10km run!",
    "I need to understand quantum entanglement.",
    "I feel really calm and content today.",
]

def make_prompt(user_input: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

dataset = Dataset.from_list([{"prompt": make_prompt(p), "raw_input": p} for p in RAW_PROMPTS])
print(f"Built {len(dataset)} GRPO prompts.")


Built 20 GRPO prompts.


### Cell 5 -- Reward functions

Three signals, same shape as the original three, but each one now checks against something real instead of a word list:
1. **Format** -- is `<state>` well-formed and parseable?
2. **Calibration** -- does the model's self-reported (V, A) match the classifier's independent reading of the *user's message*?
3. **Alignment** -- does the classifier's reading of the model's *own reply* match what it just claimed to have appraised?

Both 2 and 3 are smooth, continuous distances -- much harder to game than keyword hits.

In [5]:
import re

STATE_RE = re.compile(
    r"<state>\s*Valence:\s*([0-9]*\.?[0-9]+)\s*,\s*Arousal:\s*([0-9]*\.?[0-9]+)\s*</state>",
    re.IGNORECASE,
)

def extract_state(text: str):
    m = STATE_RE.search(text)
    if not m:
        return None
    v, a = float(m.group(1)), float(m.group(2))
    if not (0.0 <= v <= 1.0 and 0.0 <= a <= 1.0):
        return None
    return v, a

def get_text(comp):
    return comp[0]["content"] if isinstance(comp, list) else comp

def reward_format(completions, **kwargs) -> list[float]:
    scores = []
    for comp in completions:
        text = get_text(comp)
        if extract_state(text) is not None:
            scores.append(1.0)
        elif "<state>" in text and "</state>" in text:
            scores.append(0.2)   # tags present but malformed content
        else:
            scores.append(-1.0)
    return scores

def reward_calibration(prompts, completions, raw_input=None, **kwargs) -> list[float]:
    """Does the model's self-report match an external classifier's read of the USER message?"""
    scores = []
    inputs = raw_input if raw_input is not None else prompts
    for comp, user_text in zip(completions, inputs):
        text = get_text(comp)
        state = extract_state(text)
        if state is None:
            scores.append(-1.0)
            continue
        v_pred, a_pred = state
        v_true, a_true = measure_affect(user_text)
        dist = (abs(v_pred - v_true) + abs(a_pred - a_true)) / 2.0
        scores.append(1.0 - 2.0 * dist)  # 1.0 for a perfect match, negative for a bad one
    return scores

def reward_alignment(completions, **kwargs) -> list[float]:
    """Does the tone of the model's OWN reply match what it just claimed to appraise?"""
    scores = []
    for comp in completions:
        text = get_text(comp)
        state = extract_state(text)
        if state is None:
            scores.append(-1.0)
            continue
        v_pred, a_pred = state
        reply = STATE_RE.sub("", text).strip()
        if not reply:
            scores.append(-1.0)
            continue
        v_actual, a_actual = measure_affect(reply)
        dist = (abs(v_pred - v_actual) + abs(a_pred - a_actual)) / 2.0
        scores.append(1.0 - 2.0 * dist)
    return scores

print("Reward functions defined.")


Reward functions defined.


### Cell 6 -- SFT warm-start (teaches the *format*, before RL has to)

This is the fix for the run that stayed pinned at reward -3.0 for all 30 steps: GRPO's reward is entirely gated on the model emitting the exact `<state>Valence: X.X, Arousal: X.X</state>` pattern, and a base 1B instruct model has essentially zero chance of sampling that by chance. With every completion scoring the same floor, there's no contrast for GRPO's group-relative advantage to learn from -- it's not that training was slow, it's that there was no gradient at all. A short supervised warm-start fixes this by giving the format non-trivial probability *before* RL has to discover it from scratch.

The warm-start targets are generated from the same `measure_affect()` classifier used in the reward functions (not one fixed canned line repeated for every example) -- so this phase teaches the format *and* a reasonable calibration prior together, and GRPO afterward is refining an already-sane starting point rather than bootstrapping from nothing.

In [6]:
from trl import SFTConfig, SFTTrainer

# Four short reply templates bucketed by (valence, arousal) quadrant -- enough variety that the
# model can't just memorize one fixed string, without needing an LLM call to write bespoke replies.
REPLY_TEMPLATES = {
    ("high", "high"): "I'm right there with you -- that sounds intense!",
    ("high", "low"):  "That's wonderful, I'm glad things feel good and settled.",
    ("low", "high"):  "That sounds really hard and overwhelming right now.",
    ("low", "low"):   "I hear you. That sounds heavy and tiring.",
}

def _bucket(x: float) -> str:
    return "high" if x >= 0.5 else "low"

sft_rows = []
for p in RAW_PROMPTS:
    v, a = measure_affect(p)
    reply = REPLY_TEMPLATES[(_bucket(v), _bucket(a))]
    target = f"<state>Valence: {v:.1f}, Arousal: {a:.1f}</state> {reply}"
    prompt_text = make_prompt(p)
    sft_rows.append({"text": prompt_text + target + tokenizer.eos_token})

sft_dataset = Dataset.from_list(sft_rows)

sft_config = SFTConfig(
    output_dir="./affect_sft_warmup",
    per_device_train_batch_size=4,
    num_train_epochs=3,       # tiny dataset (20 rows) -> a few epochs is seconds, not minutes
    learning_rate=2e-4,       # standard LoRA warm-start LR, higher than the GRPO phase on purpose
    logging_steps=5,
    save_strategy="no",
    fp16=True,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
)

print("Running short SFT warm-start (format + calibration prior)...")
sft_trainer.train()
print("SFT warm-start complete -- GRPO now starts from a model that can already emit valid <state> tags.")


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/20 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Running short SFT warm-start (format + calibration prior)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 3 | Total steps = 9
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.984173


SFT warm-start complete -- GRPO now starts from a model that can already emit valid <state> tags.


### Cell 7 -- Train with GRPO

`num_generations=4` (was 2) so GRPO's group-relative advantage has real contrast to learn from. `per_device_train_batch_size` is set to match `num_generations` directly (Unsloth was silently bumping this up itself before, with a warning). Sequence lengths are kept short on purpose -- that's most of your speed budget. Start with `max_steps=60` and watch the time-per-step in the logs; scale it down if you're going to run over ~25 minutes, to leave time for saving.

In [7]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="./affect_grpo_output",
    learning_rate=5e-6,
    per_device_train_batch_size=4,   # matches num_generations directly, avoids Unsloth auto-bumping it
    gradient_accumulation_steps=1,
    num_generations=4,
    max_prompt_length=256,
    max_completion_length=120,
    max_steps=60,          # adjust down if step time in the logs pushes you past your time budget
    logging_steps=5,
    save_strategy="no",
    fp16=True,              # T4 doesn't have solid bf16 tensor-core support; fp16 is the right call here
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_format, reward_calibration, reward_alignment],
    args=grpo_config,
    train_dataset=dataset,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete!")


Starting GRPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureW

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_format / mean,rewards / reward_format / std,rewards / reward_calibration / mean,rewards / reward_calibration / std,rewards / reward_alignment / mean,rewards / reward_alignment / std
5,-0.150758,0.861779,1.795078,37.550000,23.800000,53.000000,0.000000,37.550000,23.800000,53.000000,0.541941,0.720000,0.469033,0.177079,0.623004,-0.035299,0.803779
10,0.079268,-0.025060,1.879971,45.500000,31.200000,61.600000,0.000000,45.500000,31.200000,61.600000,0.538499,0.500000,0.659474,-0.176007,0.701856,-0.349054,0.692751
15,0.032498,0.578660,1.938202,65.900000,32.600000,89.200000,0.150000,57.150000,32.600000,78.800000,0.326431,0.580000,0.603316,0.010803,0.699018,-0.012143,0.668585
20,0.063979,0.165499,1.264039,58.250000,28.600000,88.600000,0.100000,52.100000,28.600000,80.200000,0.452979,0.500000,0.372376,-0.132896,0.474552,-0.201605,0.495095
25,-0.056009,0.638180,1.490049,53.600000,30.800000,74.800000,0.000000,53.600000,30.800000,74.800000,0.433942,0.740000,0.452376,-0.036883,0.522818,-0.064937,0.533280
30,-0.197015,0.651666,1.607371,46.000000,27.200000,68.000000,0.000000,46.000000,27.200000,68.000000,0.431827,0.640000,0.433957,0.106552,0.608774,-0.094885,0.674836
35,-0.089164,1.880859,0.873111,67.800000,40.600000,87.200000,0.200000,59.450000,40.600000,77.200000,0.436576,0.960000,0.080000,0.580177,0.292675,0.340681,0.581656
40,0.061635,0.490365,0.793820,52.750000,27.000000,82.800000,0.050000,49.716667,27.000000,75.400000,0.523459,0.640000,0.309033,-0.014593,0.246135,-0.135042,0.283272
45,-0.066097,0.706119,1.701541,49.650000,30.000000,75.600000,0.050000,46.650000,30.000000,65.400000,0.453635,0.780000,0.440000,0.061414,0.643182,-0.135295,0.722054
50,-0.092877,1.442644,1.047997,53.300000,38.800000,75.000000,0.050000,51.266667,38.800000,67.800000,0.439686,0.880000,0.172376,0.316827,0.436753,0.245818,0.458419


Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=

Training complete!


### Cell 8 -- Save

**Adapter save is the fast, default path** -- seconds, not minutes, and there's no vocab-resize mismatch to worry about since we never touched the tokenizer's vocab. Full GGUF export (unquantize -> merge -> llama.cpp convert) is real but can take well past what's left of a 30-minute session -- treat it as an optional step to run separately once you're happy with the adapter, not part of the timed run.

In [8]:
SAVE_DIR = "affect_lora_adapter"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"LoRA adapter saved to ./{SAVE_DIR} (fast path).")

# --- Optional, slower, run separately if you have time left ---
model.save_pretrained_gguf(
    "affect_model_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)


Unsloth: Restored added_tokens_decoder metadata in affect_lora_adapter/tokenizer_config.json.


LoRA adapter saved to ./affect_lora_adapter (fast path).
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in affect_model_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:07<00:00,  7.79s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:11<00:00, 11.44s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/affect_model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9964-mix-53618c5 (app-b9964-mix-53618c5-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['affect_model_gguf_gguf/Llama-3.2-1B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions

{'save_directory': 'affect_model_gguf',
 'gguf_directory': 'affect_model_gguf_gguf',
 'gguf_files': ['affect_model_gguf_gguf/Llama-3.2-1B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'affect_model_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}